<a href="https://colab.research.google.com/github/twillixa/SL/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import, Load and Cleaning


In [21]:
import os

# --- CONFIG: update these two lines ---
GITHUB_REPO_URL = "https://github.com/twillixa/SL.git"
CSV_PATH        = "C house price.csv"
# --------------------------------------

REPO_NAME = GITHUB_REPO_URL.split("/")[-1].replace(".git", "")

# Clone only if not already cloned (avoids errors on re-run)
if not os.path.exists(f"/content/{REPO_NAME}"):
    os.system(f"git clone {GITHUB_REPO_URL}")
    print(f"Repo cloned: {REPO_NAME}")
else:
    print(f"Repo already cloned, pulling latest changes...")
    os.system(f"cd /content/{REPO_NAME} && git pull")

# Change working directory to repo
os.chdir(f"/content/{REPO_NAME}")
print(f"Working directory: {os.getcwd()}")

# Load dataset
import pandas as pd
df = pd.read_excel("distance_matrix_1.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Repo already cloned, pulling latest changes...
Working directory: /content/SL
Dataset loaded -- 31 rows x 32 columns


,DISTANCE MATRIX,"Av. Bergières 10, 1004 Lausanne","Route de Lavaux 36, 1095 Lutry","Route de la Conversion 261, 1093 La Conversion","Av. des Boveresses 44, 1010 Lausanne","Chemin de la Colice 24, 1023 Crissier","Route de Cossonay 30, 1023 Crissier","Chemin des Lentillières 15, 1023 Crissier","Chemin de Mongevon 28, 1023 Crissier","Chemin de la Gottrause, 11, 1023 Crissier",...,"Route de Genève 25, 1180 Rolle","Rte de Crassier 9, 1262 Eysins","Av. Reverdil 6, 1260 Nyon","Rue de la Morâche 9, 1260 Nyon","Place Bel Air 8, 1260 Nyon","Rue de la Colombière 28, 1260 Nyon","Rue César Soulié 3, 1260 Nyon","Chemin de la Vuarpillière 7, 1260 Nyon","Chemin Des Chalets 9, 1279 Chavannes-de-Bogis","Rue Blavignac 17, 1227 Carouge"
0,"Av. Bergières 10, 1004 Lausanne",0.000,6.276,7.195,7.688,6.372,6.370,5.585,5.777,6.520,...,29.320,41.936,42.259,42.096,42.632,42.708,41.096,42.001,47.538,75.164
1,"Route de Lavaux 36, 1095 Lutry",6.276,0.000,1.683,10.289,13.277,13.274,12.429,12.226,12.661,...,32.935,45.551,45.874,45.711,46.248,46.324,44.711,45.616,51.153,78.246
2,"Route de la Conversion 261, 1093 La Conversion",7.195,1.683,0.000,8.605,17.877,17.874,17.029,16.826,17.261,...,39.509,52.125,52.447,52.284,52.821,52.897,51.285,52.189,57.727,84.373
3,"Av. des Boveresses 44, 1010 Lausanne",7.688,10.289,8.605,0.000,14.458,14.455,13.609,13.406,13.842,...,36.090,48.705,49.028,48.865,49.402,49.478,47.865,48.770,54.307,81.475
4,"Chemin de la Colice 24, 1023 Crissier",6.372,13.277,17.877,14.458,0.000,0.235,1.485,1.678,2.065,...,25.666,38.281,38.604,38.441,38.978,39.054,37.441,38.346,43.883,71.051


## Import of required librairies




In [22]:
pip install ortools

In [11]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

# ---------- Load distance matrix ----------
file_path = "distance_matrix_1.xlsx"   # change this

df = pd.read_excel("distance_matrix_1.xlsx", index_col=0)
distance_matrix = df.to_numpy().tolist()
locations = df.index.tolist()

# ---------- Choose fixed start and end ----------
start_node = 0   # change this
end_node = len(distance_matrix) - 1   # change this

data = {}
data["distance_matrix"] = distance_matrix
data["num_vehicles"] = 1
data["starts"] = [start_node]
data["ends"] = [end_node]

manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),
    data["num_vehicles"],
    data["starts"],
    data["ends"]
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data["distance_matrix"][from_node][to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 10

solution = routing.SolveWithParameters(search_parameters)

if solution:
    index = routing.Start(0)
    route = []
    route_distance = 0

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(locations[node])
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += routing.GetArcCostForVehicle(previous_index, index, 0)

    route.append(locations[manager.IndexToNode(index)])

    print("Open route:")
    print(" -> ".join(route))
    print("Total distance:", route_distance)
else:
    print("No solution found.")

Open route:
Av. Bergières 10, 1004 Lausanne -> Route de Lavaux 36, 1095 Lutry -> Route de la Conversion 261, 1093 La Conversion -> Av. des Boveresses 44, 1010 Lausanne -> Route de la Maladière 74, 1022 Chavannes-près-Renens -> EPFL Innovation Park Building C, 1015 Lausanne -> Route de Vallaire 149, 1024 Ecublens VD -> Chemin du Croset 9B, 1024 Ecublens VD -> Chemin de la Gottrause, 11, 1023 Crissier -> Chemin de Mongevon 28, 1023 Crissier -> Chemin des Lentillières 15, 1023 Crissier -> Chemin de la Colice 24, 1023 Crissier -> Route de Cossonay 30, 1023 Crissier -> Rue de l'Industrie 63, 1030 Bussigny -> Rue des Artisans 10, 1026 Echandens -> Rte de Denges 28E, 1027 Lonay -> Rue de Lausanne 45, 1110 Morges -> Rue des Charpentiers 4, 1110 Morges -> Route de la Longeraie 7, 1110 Morges -> Route de Lully 5C, 1131 Tolochenaz -> Chemin Lucien Chevallaz 5, 1170 Aubonne -> Route de Genève 25, 1180 Rolle -> Rue César Soulié 3, 1260 Nyon -> Rue de la Colombière 28, 1260 Nyon -> Place Bel Air 8, 

In [12]:
df = pd.read_excel("distance_matrix_2.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 32 rows x 33 columns


,DISTANCE MATRIX,"Av. Bergières 10, 1004 Lausanne, Switzerland, 1004, ,","Chemin de Maillefer 59-61, 1052, Le Mont-sur-Lausanne, CH","Avenue Gratta Paille 2, 1018, Lausanne, CH","Chemin des Avelines 5, 1004, Lausanne, CH","Rte. des Flumeaux 1, 1008, Prilly, CH","Ch. de la Roche 12, 1020, Renens, CH","Rue neuve 3-5, 1020, Renens VD, CH","Avenue d'Epenex, 4C 1er Etage, 1020, Renens VD, CH","Rue du Marais 17, 1020, Renens VD, CH",...,"Place de la Riponne 10, 1014, Lausanne, CH","Rue César-Roux 31, 1001, Lausanne, CH","Rue Caroline 23, 1003, Lausanne, CH","Rue du Bugnon 21, 1011, Lausanne, CH","Av. du Tribunal-Fédéral 34, 1005, Lausanne, CH","Avenue Guillemin 16, 1009, Pully, CH","Av. de Lavaux 65, 1009, Pully, CH","Av. Charles Ferdinand Ramuz 60, 1009, Pully, CH","Avenue de la gare, 6, 1003, Lausanne, CH","Avenue Georgette 6, 1003, Lausanne, CH"
0,"Av. Bergières 10, 1004 Lausanne, Switzerland, ...",0.00,2.89,1.58,1.03,2.04,3.04,4.48,3.75,3.75,...,1.31,1.93,2.14,2.62,2.63,3.65,4.28,3.91,2.48,2.66
1,"Chemin de Maillefer 59-61, 1052, Le Mont-sur-L...",2.89,0.00,1.67,2.90,4.20,4.91,6.35,5.62,5.62,...,2.80,3.17,3.38,3.87,3.97,5.09,5.72,5.42,4.65,3.93
2,"Avenue Gratta Paille 2, 1018, Lausanne, CH",1.57,1.66,0.00,1.57,2.59,3.59,5.02,4.30,4.30,...,2.70,3.32,3.53,4.01,4.02,5.04,5.67,5.12,3.69,3.87
3,"Chemin des Avelines 5, 1004, Lausanne, CH",1.03,3.15,1.83,0.00,1.27,2.14,3.58,2.85,2.85,...,2.16,2.78,2.99,3.47,3.48,4.50,5.13,4.39,2.96,3.14
4,"Rte. des Flumeaux 1, 1008, Prilly, CH",2.02,3.81,2.49,1.20,0.00,1.13,2.35,1.58,1.58,...,3.15,3.78,3.98,4.47,4.05,5.07,5.70,5.23,3.80,3.98


In [19]:
# ---------- Load distance matrix from Excel ----------
file_path = "distance_matrix_2.xlsx"

df = pd.read_excel("distance_matrix_2.xlsx", index_col=0)
distance_matrix = df.to_numpy().tolist()
locations = df.index.tolist()

# ---------- Data model ----------
data = {}
data["distance_matrix"] = distance_matrix
data["num_vehicles"] = 1
data["depot"] = 0   # starting/ending node index

# ---------- Routing model ----------
manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),
    data["num_vehicles"],
    data["depot"]
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data["distance_matrix"][from_node][to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# ---------- Search parameters ----------
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 10

# ---------- Solve ----------
solution = routing.SolveWithParameters(search_parameters)

# ---------- Print solution ----------
if solution:
    index = routing.Start(0)
    route = []
    route_distance = 0

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(locations[node])
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += routing.GetArcCostForVehicle(previous_index, index, 0)

    route.append(locations[manager.IndexToNode(index)])

    print("Route:")
    print(" -> ".join(route))
    print("Total distance:", route_distance)
else:
    print("No solution found.")

Route:
Av. Bergières 10, 1004 Lausanne, Switzerland, 1004 -> Rue de la Borde 41, 1018, Lausanne, CH -> Place de la Riponne 10, 1014, Lausanne, CH -> Place Bel-Air 1, 1003, Lausanne, CH -> Avenue Georgette 6, 1003, Lausanne, CH -> Avenue de la gare, 6, 1003, Lausanne, CH -> Rue Caroline 23, 1003, Lausanne, CH -> Rue du Bugnon 21, 1011, Lausanne, CH -> Rue César-Roux 31, 1001, Lausanne, CH -> Av. du Tribunal-Fédéral 34, 1005, Lausanne, CH -> Avenue Guillemin 16, 1009, Pully, CH -> Av. de Lavaux 65, 1009, Pully, CH -> Av. Charles Ferdinand Ramuz 60, 1009, Pully, CH -> Place de la navigation 14, 1006, Lausanne, CH -> Avenue de Rhodanie 46b, 1007, Lausanne, CH -> Avenue de Rhodanie 40a, 1007, Lausanne, CH -> Chemin de Primerose 11, 1007, Lausanne, CH -> Avenue William-Fraisse 3, 1006, Lausanne, CH -> Avenue d'Ouchy 4, 1006, Lausanne, CH -> Av. Louis-Ruchonnet 16, 1003, Lausanne, CH -> Place de la Gare 11a, 1003, Lausanne, CH -> Avenue de la Gare 44, 1003, Lausanne, CH -> Avenue de Sévelin 3

In [26]:
df = pd.read_excel("distance_matrix_3.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 31 rows x 32 columns


,DISTANCE MATRIX,"Rue Blavignac 17, 1227, ,","Rue Blavignac, 10, 1227, Carouge, CH","Rte de Saint Julien 7, 1227, Carouge GE, CH","Rue Joseph Girard 18, 1227, Carouge GE, CH","Clos de la Fonderie 1, 1227, Carouge GE, CH","Avenue de la Praille 26, 1227, Genève, CH","Avenue de la Praille 50, 1227, Carouge GE, CH","Esplanade de Pont Rouge 9c, 1212, Grand Lancy, CH","Rte des Jeunes 9, 1227, Les Acacias, CH",...,"Rue Rodolphe-Toepffer 11 bis, 1206, Genève, CH","Boulevard Helvetique, 8, 1205, Genève, CH","Place de la Taconnerie 3, 1204, Genève, CH","Grand-Rue 25, 1204, Genève, CH","Rue du Port 12, 1204, Genève, CH","Rue Robert-Céard 8, 1204, Genève, CH","Rue du Rhône 62, 1204, Genève, CH","Place du Molard 3, 1204, Genève, CH","Rue du Rhône 42, 1204, Genève, CH","12 Place de la Fusterie, 1204, Genève, CH"
0,"Rue Blavignac 17, 1227, ,",0.0,0.23,1.57,1.32,"1,4",0.94,1.46,"2,5","1,6",...,2.93,"2,8",3.41,3.23,3.69,3.91,"3,9",3.98,"4,09",3.45
1,"Rue Blavignac, 10, 1227, Carouge, CH",0.19,0.0,1.58,1.33,"1,4",0.94,1.46,2.51,"1,6",...,2.94,"2,8",3.41,3.23,"3,7",3.91,3.91,3.98,"4,1",3.45
2,"Rte de Saint Julien 7, 1227, Carouge GE, CH",1.21,"1,03",0.0,0.69,1.71,1.42,1.94,1.69,"2,07",...,3.27,2.95,3.61,3.57,3.84,"4,06","4,05",4.13,4.24,3.78
3,"Rue Joseph Girard 18, 1227, Carouge GE, CH",1.16,0.97,"1,12",0.0,"1,05",1.29,1.81,"2,05",1.95,...,2.96,2.29,2.96,"2,07",3.19,"3,4","3,4",3.47,3.59,3.67
4,"Clos de la Fonderie 1, 1227, Carouge GE, CH",1.68,"1,5",2.25,"1,6",0.0,0.87,1.38,3.21,1.52,...,"2,04",1.73,"2,4",2.34,2.63,2.84,2.84,2.91,"3,03",2.55


In [25]:
# ---------- Load distance matrix from Excel ----------
file_path = "distance_matrix_3.xlsx"

df = pd.read_excel("distance_matrix_3.xlsx", index_col=0)
distance_matrix = df.to_numpy().tolist()
locations = df.index.tolist()

# ---------- Data model ----------
data = {}
data["distance_matrix"] = distance_matrix
data["num_vehicles"] = 1
data["depot"] = 0   # starting/ending node index

# ---------- Routing model ----------
manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),
    data["num_vehicles"],
    data["depot"]
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data["distance_matrix"][from_node][to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# ---------- Search parameters ----------
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 10

# ---------- Solve ----------
solution = routing.SolveWithParameters(search_parameters)

# ---------- Print solution ----------
if solution:
    index = routing.Start(0)
    route = []
    route_distance = 0

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(locations[node])
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += routing.GetArcCostForVehicle(previous_index, index, 0)

    route.append(locations[manager.IndexToNode(index)])

    print("Route:")
    print(" -> ".join(route))
    print("Total distance:", route_distance)
else:
    print("No solution found.")

Route:
Rue Blavignac 17, 1227, , -> 12 Place de la Fusterie, 1204, Genève, CH -> Rue du Rhône 42, 1204, Genève, CH -> Place du Molard 3, 1204, Genève, CH -> Rue du Rhône 62, 1204, Genève, CH -> Rue Robert-Céard 8, 1204, Genève, CH -> Rue du Port 12, 1204, Genève, CH -> Grand-Rue 25, 1204, Genève, CH -> Place de la Taconnerie 3, 1204, Genève, CH -> Boulevard Helvetique, 8, 1205, Genève, CH -> Rue Rodolphe-Toepffer 11 bis, 1206, Genève, CH -> Rue de Villereuse 22, 1207, Genève, CH -> Avenue de la Gare des Eaux-Vives 11, 1207, Genève, CH -> Avenue Rosemont 8, 1208, Genève, CH -> Chemin Frank-Thomas 32, 1208, Genève, CH -> Route de Malagnou 28, 1208, Genève, CH -> Route de Florissant 81, 1206, Genève, CH -> Avenue Peschier 41, 1206, Genève, CH -> Av. de la Roseraie 64, 1205, Genève, CH -> Rue des Battoirs 7, 1205, Genève, CH -> Route des Acacias 14, 1227, Les Acacias, CH -> Rue François-Dussaud , 1227, Les Acacias, CH -> Passage de la Radio, 1211, Genève 1, CH -> Rte des Jeunes 9, 1227, Le

In [17]:
df = pd.read_excel("distance_matrix_4.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 17 rows x 18 columns


,DISTANCE MATRIX,"Rue Blavignac 17, 1227, ,","Boulevard James Fazy 8, 1201, Genève, CH","Rue de Chantepoulet, 5, 1201, Genève, CH","Quai du Mont Blanc, 7, 1201, Genève, CH","Quai du Mont Blanc 5, 1201, Genève, CH","Quai du Mont Blanc 3, 1201, Genève, CH","Quai du Mont-Blanc 29, 1201, Genève, CH","Chemin des mines 9, 1203, Genève, CH","Avenue de Sécheron 15, 1202, Geneva, CH","Avenue de Secheron 6, 1202, Genève, CH","Chemin Eugène-Rigot 2, 1202, Genève, CH","Rue de Vermont 62, 1202, Genève, CH","Avenue de la paix 19, 1202, Genève, CH","Avenue Louis-Casai 18, 1209, Genève, CH","Rue de Bourgogne 23, 1203, Genève, CH","Rue de Saint-Jean 30, 1203, Genève, CH","Place de Pont-Rouge, 1212 Lancy, Suisse"
0,"Rue Blavignac 17, 1227, Carouge, CH",0.0,3.53,3.92,4.19,4.18,4.15,4.66,5.42,5.24,"5,2",5.57,5.33,6.39,6.15,4.82,"4,1",1.98
1,"Boulevard James Fazy 8, 1201, Genève, CH",3.46,0.0,0.56,0.83,0.82,0.79,"1,3","2,06",1.87,1.84,2.21,1.71,2.78,2.93,1.59,1.0,3.15
2,"Rue de Chantepoulet, 5, 1201, Genève, CH",3.98,0.54,0.0,0.35,0.34,0.31,0.82,2.0,1.81,1.77,2.14,"1,9",2.97,2.84,"2,04",1.52,3.66
3,"Quai du Mont Blanc, 7, 1201, Genève, CH",4.26,0.82,0.37,0.0,0.02,0.04,0.53,1.99,1.81,1.77,2.14,"9,26",2.97,3.41,2.32,"8,26",3.95
4,"Quai du Mont Blanc 5, 1201, Genève, CH",4.21,0.78,0.32,0.23,0.0,0.23,0.71,2.0,1.81,1.77,2.14,"9,26",2.97,3.42,2.28,1.76,"9,26"


In [18]:
# ---------- Load distance matrix from Excel ----------
file_path = "distance_matrix_4.xlsx"

df = pd.read_excel("distance_matrix_4.xlsx", index_col=0)
distance_matrix = df.to_numpy().tolist()
locations = df.index.tolist()

# ---------- Data model ----------
data = {}
data["distance_matrix"] = distance_matrix
data["num_vehicles"] = 1
data["depot"] = 0   # starting/ending node index

# ---------- Routing model ----------
manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),
    data["num_vehicles"],
    data["depot"]
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data["distance_matrix"][from_node][to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# ---------- Search parameters ----------
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 10

# ---------- Solve ----------
solution = routing.SolveWithParameters(search_parameters)

# ---------- Print solution ----------
if solution:
    index = routing.Start(0)
    route = []
    route_distance = 0

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(locations[node])
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += routing.GetArcCostForVehicle(previous_index, index, 0)

    route.append(locations[manager.IndexToNode(index)])

    print("Route:")
    print(" -> ".join(route))
    print("Total distance:", route_distance)
else:
    print("No solution found.")

Route:
Rue Blavignac 17, 1227, Carouge, CH -> Place de Pont-Rouge, 1212 Lancy, Suisse -> Rue de Saint-Jean 30, 1203, Genève, CH -> Rue de Bourgogne 23, 1203, Genève, CH -> Avenue Louis-Casai 18, 1209, Genève, CH -> Avenue de la paix 19, 1202, Genève, CH -> Rue de Vermont 62, 1202, Genève, CH -> Chemin Eugène-Rigot 2, 1202, Genève, CH -> Avenue de Secheron 6, 1202, Genève, CH -> Avenue de Sécheron 15, 1202, Geneva, CH -> Chemin des mines 9, 1203, Genève, CH -> Quai du Mont-Blanc 29, 1201, Genève, CH -> Quai du Mont Blanc 3, 1201, Genève, CH -> Quai du Mont Blanc 5, 1201, Genève, CH -> Quai du Mont Blanc, 7, 1201, Genève, CH -> Rue de Chantepoulet, 5, 1201, Genève, CH -> Boulevard James Fazy 8, 1201, Genève, CH -> Rue Blavignac 17, 1227, Carouge, CH
Total distance: 0


In [27]:
df = pd.read_excel("distance_matrix_5.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 23 rows x 24 columns


,DISTANCE MATRIX,"Rue Blavignac 17, 1227, ,","Ch. de la Place-Verte 34, 1234, Vessy, CH","Chemin du Pré-Fleuri 15, 1228, Plan-les-Ouates, CH","Chemin Charles-Borgeaud 27, 1213, Onex, CH","Rte de Chancy 85, 1213, Onex, CH","Avenue des Morgines, 8, 1213, Genève, CH","Chemin du Chateau-Bloch 11, 1219, Le Lignon, CH","Avenue de l’Etang 55, 1219, Châtelaine, CH","Chemin de Blandonnet 2, 1214, Vernier, CH",...,"Route de Satigny 50, 1214, Vernier, CH","Route de Satigny 52, 1242, Satigny, CH","Losinger Quarz'Up Chemin Delay, 7, 1214, Vernier, CH","25b route du Bois-de-Bay, 1242, Satigny, CH","Rue de la Maison Carrée 30, 1242, Satigny, CH","Route de l'Aéroport, 21, 1215, Genève, CH","Route de Pré-Bois 29, 1216, Cointrin, CH","L'Ancienne Route 17A, 1218, Le Grand-Saconnex, CH","Route de suisse 160, 1290, Versoix, CH","Ch. Jean-Baptiste Vandelle 3A, 1290, Versoix, CH"
0,"Rue Blavignac 17, 1227, ,",0.0,4.25,3.86,4.19,"3,7",3.42,6.64,13.16,12.74,...,13.51,"16,9","14,02",15.41,15.93,14.68,13.15,8.39,23.95,23.15
1,"Ch. de la Place-Verte 34, 1234, Vessy, CH",3.95,0.0,"6,2",7.42,"7,06",6.86,8.61,16.54,"16,12",...,16.88,20.27,17.39,18.78,"19,3","18,05",16.53,"8,7","14,9","14,1"
2,"Chemin du Pré-Fleuri 15, 1228, Plan-les-Ouates...",3.92,6.18,0.0,3.54,"4,12","3,9",10.14,9.48,"9,06",...,9.82,13.21,10.33,9.72,10.23,10.99,9.47,11.32,20.27,19.46
3,"Chemin Charles-Borgeaud 27, 1213, Onex, CH",4.66,7.82,3.54,0.0,"1,02",1.75,4.83,5.25,"8,1",...,8.86,12.44,9.37,9.26,9.78,"10,04",8.51,6.58,19.31,"18,5"
4,"Rte de Chancy 85, 1213, Onex, CH",4.63,6.64,3.72,1.22,0.0,0.99,3.81,4.23,4.75,...,5.51,9.61,"6,02",9.68,"10,2","7,04",5.44,5.56,16.32,15.51
